In [0]:
dbutils.widgets.text(name="env", defaultValue="", label ="Enter environment")
env = dbutils.widgets.get("env")

In [0]:
%run "./commons"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
    StructField("PatientID",StringType()),
    StructField("FirstName", StringType()),
    StructField("LastName",StringType()),
    StructField("MiddleName",StringType()),
    StructField("SSN",StringType()),
    StructField("PhoneNumber",StringType()),
    StructField("Gender",StringType()),
    StructField("DOB",StringType()),
    StructField("Address",StringType()),
    StructField("ModifiedDate",StringType()),
    StructField("Extract_Time", StringType()),
    StructField("filename", StringType()),
    StructField("transformed_time", TimestampType())
])

def read_silver_patient(env):
    print("reading silver patient table")
    df_silver = spark.readStream.schema(schema).table(f"{env}_catalog.silver.patient")
    return df_silver

def count_patient_occurence(df):
    print("counting patient occurence records")
    df_agg = df.groupBy("PatientId").count()
    #df_count = df.join(df_agg,"PatientId")
    return df_agg

def create_load_time(df):
    print("adding extra column to track load time")
    df = df.withColumn("load_time", current_timestamp())
    return df

def write_to_gold_patient(df, env):
    print("writing to gold patient table")
    write_stream = df.writeStream.format("delta").option("checkpointLocation", checkpoint_path + "/gold_checkpoint/").queryName("gold_patient_load").trigger(availableNow=True).outputMode("complete").toTable(f"{env}_catalog.gold.patient")

    write_stream.awaitTermination()
    print("loaded Gold table successfully")

In [0]:
#main execution
df_silver = read_silver_patient(env)
print("read the silver patient")
df_count = count_patient_occurence(df_silver)
print("patient occurence count calculated")
df = create_load_time(df_count)
print("load time is created")
write_to_gold_patient(df,env)
print("Gold load is completed")